### Импорты, seed и среда

In [1]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import sys
import random
from typing import List, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

import faiss
import sklearn
import torch

import sentence_transformers
from sentence_transformers import SentenceTransformer

In [3]:
print("faiss:", faiss.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("torch:", torch.__version__)

faiss: 1.13.2
sentence-transformers: 5.4.0
torch: 2.10.0+cpu


In [4]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

In [5]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


In [6]:
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 100)

In [7]:
os.makedirs("artifacts", exist_ok=True)

### База знаний и первичный анализ

In [21]:
data_path = "data"
docs_list = []

if os.path.exists(data_path):
    for filename in os.listdir(data_path):
        if filename.endswith(".txt") and filename.startswith("rag_"):
            file_path = os.path.join(data_path, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
                doc_name = os.path.splitext(filename)[0]
                docs_list.append({"doc_id": doc_name, "text": content})

df_docs = pd.DataFrame(docs_list)

In [22]:
print(f"Загружено документов: {len(docs_list)}")
display(df_docs)

Загружено документов: 6


,doc_id,text
0,rag_advantages,Использование RAG предоставляет ряд существенных преимуществ для улучшения больших языковых моде...
1,rag_applications,Благодаря своей способности сочетать обширные знания больших языковых моделей с актуальной инфор...
2,rag_paradigms,"Развитие методов генерации с дополненной выборкой начинается с базовой парадигмы Naive RAG, осно..."
3,rag_abstract,"Генерация с дополненной выборкой (англ. Retrieval-Augmented Generation, RAG) — это подход, при к..."
4,rag_evaluation,Интеграция внешних данных в большие языковые модели посредством технологии RAG сталкивается с фа...
5,rag_challenges,"Несмотря на многочисленные преимущества, применение RAG в больших языковых моделях сопряжено с р..."


In [23]:
for i, row in df_docs.head(3).iterrows():
    display(Markdown(f"#### Документ: {row['doc_id']}.txt"))
    display(Markdown(f"{row['text'][:500]}..."))

#### Документ: rag_advantages.txt

Использование RAG предоставляет ряд существенных преимуществ для улучшения больших языковых моделей. Повышенная точность и надежность: RAG обеспечивает доступ LLM к самым актуальным и надежным фактам, что значительно снижает вероятность генерации неверного или вводящего в заблуждение контента. За счет обоснования ответов на внешних, проверенных источниках информации, RAG гарантирует, что LLM оперирует наиболее точными данными. Уменьшение галлюцинаций: Одним из ключевых преимуществ RAG является м...

#### Документ: rag_applications.txt

Благодаря своей способности сочетать обширные знания больших языковых моделей с актуальной информацией из внешних источников, RAG находит широкое применение в самых разных областях. RAG находит применение во множестве контекстов, включая вопросно-ответные системы, анализ временных рядов с использованием агентных RAG-фреймворков, повышение фактичности в медицинских системах обработки изображений и языка, улучшение точности в юридических и политических приложениях, повышение качества автоматическо...

#### Документ: rag_paradigms.txt

Развитие методов генерации с дополненной выборкой начинается с базовой парадигмы Naive RAG, основанной на простом цикле индексации, извлечения и генерации. Парадигма Advanced RAG внедряет этапы предварительной и последующей обработки извлекаемой информации, направленных на повышение качества и релевантности контекста. Современный этап, парадигма Modular RAG, представляет собой гибкую и адаптивную архитектуру, позволяющую интегрировать новые функциональные модули, изменять паттерны взаимодействия...

### Чанкинг документов

In [24]:
def split_text_into_chunks(text: str, chunk_size_sentences: int = 5, overlap_sentences: int = 1) -> List[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    chunks = []
    if not sentences:
        return chunks

    step = chunk_size_sentences - overlap_sentences
    if step <= 0:
        step = 1

    for i in range(0, len(sentences), step):
        chunk = " ".join(sentences[i : i + chunk_size_sentences])
        chunks.append(chunk)
        if i + chunk_size_sentences >= len(sentences):
            break

    return chunks

In [25]:
example_text = df_docs.iloc[0]['text']
example_chunks = split_text_into_chunks(example_text, chunk_size_sentences=3, overlap_sentences=1)

print(f"Количество чанков: {len(example_chunks)}")
print("Пример первого чанка:")
display(Markdown(example_chunks[0]))
print("Пример второго чанка:")
display(Markdown(example_chunks[1]))

Количество чанков: 6
Пример первого чанка:


Использование RAG предоставляет ряд существенных преимуществ для улучшения больших языковых моделей. Повышенная точность и надежность: RAG обеспечивает доступ LLM к самым актуальным и надежным фактам, что значительно снижает вероятность генерации неверного или вводящего в заблуждение контента. За счет обоснования ответов на внешних, проверенных источниках информации, RAG гарантирует, что LLM оперирует наиболее точными данными.

Пример второго чанка:


За счет обоснования ответов на внешних, проверенных источниках информации, RAG гарантирует, что LLM оперирует наиболее точными данными. Уменьшение галлюцинаций: Одним из ключевых преимуществ RAG является минимизация риска «галлюцинаций» — генерации LLMs вымышленной или неточной информации. Предоставляя LLM конкретные, извлеченные факты, RAG значительно снижает вероятность создания ответов, не основанных на реальных данных.

In [26]:
chunks_data = []

for _, row in df_docs.iterrows():
    doc_id = row['doc_id']
    text = row['text']

    text_chunks = split_text_into_chunks(text, chunk_size_sentences=3, overlap_sentences=1)

    for i, chunk_content in enumerate(text_chunks):
        chunks_data.append({
            "doc_id": doc_id,
            "chunk_id": f"chunk_{i+1:02d}",
            "chunk_text": chunk_content
        })

df_chunks = pd.DataFrame(chunks_data)

print(f"Всего создано чанков: {len(df_chunks)}")
display(df_chunks.head(10))

Всего создано чанков: 71


,doc_id,chunk_id,chunk_text
0,rag_advantages,chunk_01,Использование RAG предоставляет ряд существенных преимуществ для улучшения больших языковых моде...
1,rag_advantages,chunk_02,"За счет обоснования ответов на внешних, проверенных источниках информации, RAG гарантирует, что ..."
2,rag_advantages,chunk_03,"Предоставляя LLM конкретные, извлеченные факты, RAG значительно снижает вероятность создания отв..."
3,rag_advantages,chunk_04,"Это обеспечивает LLMs возможность предоставлять ответы, основанные на самой последней доступной ..."
4,rag_advantages,chunk_05,Возможность проверить источники информации позволяет пользователям убедиться в достоверности пол...
5,rag_advantages,chunk_06,RAG обеспечивает более экономичный способ адаптации LLMs к новым данным и доменам знаний. Интегр...
6,rag_applications,chunk_01,Благодаря своей способности сочетать обширные знания больших языковых моделей с актуальной инфор...
7,rag_applications,chunk_02,Вопросно-ответные и консультационные системы Одним из наиболее распространенных применений RAG я...
8,rag_applications,chunk_03,Способность RAG предоставлять обоснованные ответы с цитатами и ссылками на первоисточник повышае...
9,rag_applications,chunk_04,В медицинских диагностических системах RAG может использоваться для поиска и интеграции последни...


### Эмбеддинги и индекс FAISS

In [14]:
model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(model_name, device=DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
def get_embeddings(texts: List[str]) -> np.ndarray:
    embeddings = model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
        )
    return embeddings

print("Генерация эмбеддингов для чанков...")
chunk_embeddings = get_embeddings(df_chunks['chunk_text'].tolist())

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)

print(f"Индекс FAISS создан. Размерность: {dimension}, количество векторов: {index.ntotal}")

Генерация эмбеддингов для чанков...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Индекс FAISS создан. Размерность: 384, количество векторов: 71


In [16]:
def search_top_k(query: str, k: int = 5) -> pd.DataFrame:
    query_embedding = get_embeddings([query])

    distances, indices = index.search(query_embedding, k)

    results = []
    for score, idx in zip(distances[0], indices[0]):
        if idx == -1: continue

        res = df_chunks.iloc[idx].to_dict()
        res['score'] = score
        results.append(res)

    return pd.DataFrame(results)

In [17]:
def display_results(query: str, search_results: pd.DataFrame) -> None:
    display(Markdown(f"### Результаты поиска по запросу: '{query}'"))
    display(search_results)

    display(Markdown("---"))
    for _, row in search_results.iterrows():
        display(Markdown(f"**Документ:** {row['doc_id']} | **Чанк:** {row['chunk_id']} | **Score:** {row['score']:.3f}"))
        display(Markdown(row['chunk_text']))
        display(Markdown("---"))

In [28]:
query = "Что такое RAG?"
search_results = search_top_k(query, k=3)
display_results(query, search_results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### Результаты поиска по запросу: 'Что такое RAG?'

,doc_id,chunk_id,chunk_text,score
0,rag_paradigms,chunk_18,"Этот подход признает критическую важность как качества самой извлеченной информации, так и эффек...",0.593496
1,rag_applications,chunk_01,Благодаря своей способности сочетать обширные знания больших языковых моделей с актуальной инфор...,0.581984
2,rag_abstract,chunk_04,RAG сочетает в себе сильные стороны традиционных систем поиска информации (поисковые системы и с...,0.545759


---

**Документ:** rag_paradigms | **Чанк:** chunk_18 | **Score:** 0.593

...

Благодаря своей способности сочетать обширные знания больших языковых моделей с актуальной информацией из внешних источников, RAG находит широкое применение в самых разных областях. RAG находит применение во множестве контекстов, включая вопросно-ответные системы, анализ временных рядов с использованием агентных RAG-фреймворков, повышение фактичности в медицинских системах обработки изображений и языка, улучшение точности в юридических и политических приложениях, повышение качества автоматического распознавания речи (ASR), обеспечение многоязычного доступа к информации, а также в качестве инструментов помощи в написании научных работ, таких как LLM-Ref или сервисы типа OpenAI Deep Research. Вопросно-ответные и консультационные системы Одним из наиболее распространенных применений RAG является открытое вопросно-ответное взаимодействие, где системы генерируют ответы на широкий спектр вопросов, извлекая релевантную информацию и обосновывая свои ответы полученными данными.

---

**Документ:** rag_abstract | **Чанк:** chunk_04 | **Score:** 0.546

RAG сочетает в себе сильные стороны традиционных систем поиска информации (поисковые системы и системы управления базами данных) с возможностями больших языковых моделей. С появлением больших языковых моделей исследования RAG первоначально фокусировались на использовании их способности к обучению в контексте, преимущественно затрагивая этап вывода. Последующие работы углубили анализ, постепенно интегрируя методы RAG в процесс тонкой настройки LLM.

---

In [29]:
query = "В каких областях применяется RAG?"
search_results = search_top_k(query, k=3)
display_results(query, search_results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### Результаты поиска по запросу: 'В каких областях применяется RAG?'

,doc_id,chunk_id,chunk_text,score
0,rag_paradigms,chunk_18,"Этот подход признает критическую важность как качества самой извлеченной информации, так и эффек...",0.732360
1,rag_applications,chunk_01,Благодаря своей способности сочетать обширные знания больших языковых моделей с актуальной инфор...,0.719011
2,rag_abstract,chunk_04,RAG сочетает в себе сильные стороны традиционных систем поиска информации (поисковые системы и с...,0.662897


---

**Документ:** rag_paradigms | **Чанк:** chunk_18 | **Score:** 0.732

...

Благодаря своей способности сочетать обширные знания больших языковых моделей с актуальной информацией из внешних источников, RAG находит широкое применение в самых разных областях. RAG находит применение во множестве контекстов, включая вопросно-ответные системы, анализ временных рядов с использованием агентных RAG-фреймворков, повышение фактичности в медицинских системах обработки изображений и языка, улучшение точности в юридических и политических приложениях, повышение качества автоматического распознавания речи (ASR), обеспечение многоязычного доступа к информации, а также в качестве инструментов помощи в написании научных работ, таких как LLM-Ref или сервисы типа OpenAI Deep Research. Вопросно-ответные и консультационные системы Одним из наиболее распространенных применений RAG является открытое вопросно-ответное взаимодействие, где системы генерируют ответы на широкий спектр вопросов, извлекая релевантную информацию и обосновывая свои ответы полученными данными.

---

**Документ:** rag_abstract | **Чанк:** chunk_04 | **Score:** 0.663

RAG сочетает в себе сильные стороны традиционных систем поиска информации (поисковые системы и системы управления базами данных) с возможностями больших языковых моделей. С появлением больших языковых моделей исследования RAG первоначально фокусировались на использовании их способности к обучению в контексте, преимущественно затрагивая этап вывода. Последующие работы углубили анализ, постепенно интегрируя методы RAG в процесс тонкой настройки LLM.

---

In [30]:
query = "Что называется галлюцинациями?"
search_results = search_top_k(query, k=3)
display_results(query, search_results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### Результаты поиска по запросу: 'Что называется галлюцинациями?'

,doc_id,chunk_id,chunk_text,score
0,rag_advantages,chunk_02,"За счет обоснования ответов на внешних, проверенных источниках информации, RAG гарантирует, что ...",0.304674
1,rag_evaluation,chunk_14,Языковые модели часто не следуют строго инструкциям и генерируют непредсказуемые ответы. Контрфа...,0.267397
2,rag_applications,chunk_03,Способность RAG предоставлять обоснованные ответы с цитатами и ссылками на первоисточник повышае...,0.248656


---

**Документ:** rag_advantages | **Чанк:** chunk_02 | **Score:** 0.305

...

Языковые модели часто не следуют строго инструкциям и генерируют непредсказуемые ответы. Контрфактуальная устойчивость (counterfactual robustness) — определяет распознавание и игнорирование заведомо ложных данных. Оценке подлежит ситуация, при которой LLM предоставляются инструкции с предупреждениями о потенциальных рисках в полученных данных.

---

**Документ:** rag_applications | **Чанк:** chunk_03 | **Score:** 0.249

Способность RAG предоставлять обоснованные ответы с цитатами и ссылками на первоисточник повышает доверие пользователей к таким системам и улучшает качество взаимодействия. В критически важных областях, таких как медицина и юриспруденция, RAG демонстрирует значительный потенциал. В медицинских диагностических системах RAG может использоваться для поиска и интеграции последних исследований или специфических данных о пациентах для генерации точных диагностических предположений.

---

### Контрольные запросы и оценка retrieval

In [43]:
qa_benchmark = [
    {
        "query_id": "q01",
        "query": "Что такое генерация с дополненной выборкой (RAG) и в чём заключается её основная идея?",
        "relevant_doc_ids": ["rag_abstract"]
    },
    {
        "query_id": "q02",
        "query": "Какие основные этапы включает работа RAG-системы?",
        "relevant_doc_ids": ["rag_abstract"]
    },
    {
        "query_id": "q03",
        "query": "Как происходит индексация?",
        "relevant_doc_ids": ["rag_paradigms"]
    },
    {
        "query_id": "q04",
        "query": "Какие существуют техники оптимизации запроса?",
        "relevant_doc_ids": ["rag_paradigms"]
    },
    {
        "query_id": "q05",
        "query": "С какими основными проблемами и ограничениями сталкивается RAG при практическом применении?",
        "relevant_doc_ids": ["rag_challenges"]
    },
    {
        "query_id": "q06",
        "query": "В каких областях и приложениях наиболее эффективно применяется технология RAG?",
        "relevant_doc_ids": ["rag_applications"]
    },
    {
        "query_id": "q07",
        "query": "Какие показатели качества используются для оценки эффективности RAG-систем?",
        "relevant_doc_ids": ["rag_evaluation"]
    },
    {
        "query_id": "q08",
        "query": "Какие есть бенчмарки для оценки RAG?",
        "relevant_doc_ids": ["rag_evaluation"]
    },
]

In [44]:
df_benchmark = pd.DataFrame(qa_benchmark)
display(df_benchmark)

,query_id,query,relevant_doc_ids
0,q01,Что такое генерация с дополненной выборкой (RAG) и в чём заключается её основная идея?,[rag_abstract]
1,q02,Какие основные этапы включает работа RAG-системы?,[rag_abstract]
2,q03,Как происходит индексация?,[rag_paradigms]
3,q04,Какие существуют техники оптимизации запроса?,[rag_paradigms]
4,q05,С какими основными проблемами и ограничениями сталкивается RAG при практическом применении?,[rag_challenges]
5,q06,В каких областях и приложениях наиболее эффективно применяется технология RAG?,[rag_applications]
6,q07,Какие показатели качества используются для оценки эффективности RAG-систем?,[rag_evaluation]
7,q08,Какие есть бенчмарки для оценки RAG?,[rag_evaluation]


In [46]:
def evaluate_retrieval(benchmark: List[Dict], k: int = 5) -> pd.DataFrame:
    results_data = []

    for item in benchmark:
        query = item['query']
        relevant_ids = set(item['relevant_doc_ids'])

        search_res = search_top_k(query, k=k)
        retrieved_doc_ids = search_res['doc_id'].tolist()
        unique_retrieved_ids = set(search_res['doc_id'].unique())

        intersection = relevant_ids.intersection(unique_retrieved_ids)

        hit = 1 if len(intersection) > 0 else 0
        recall = len(intersection) / len(relevant_ids)

        first_chunk_text = search_res.iloc[0]['chunk_text'] if not search_res.empty else ""
        rag_answer = first_chunk_text.split('\n')[0]

        results_data.append({
            "query": query,
            "relevant_doc_ids": list(relevant_ids),
            "predicted_doc_ids": retrieved_doc_ids,
            f"hit@{k}": hit,
            f"recall@{k}": recall,
            "rag_answer": rag_answer
        })

    return pd.DataFrame(results_data)

In [47]:
evaluation_df = evaluate_retrieval(qa_benchmark, k=3)
display(evaluation_df)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,query,relevant_doc_ids,predicted_doc_ids,hit@3,recall@3,rag_answer
0,Что такое генерация с дополненной выборкой (RAG) и в чём заключается её основная идея?,[rag_abstract],"[rag_paradigms, rag_abstract, rag_evaluation]",1,1.0,"Развитие методов генерации с дополненной выборкой начинается с базовой парадигмы Naive RAG, осно..."
1,Какие основные этапы включает работа RAG-системы?,[rag_abstract],"[rag_paradigms, rag_paradigms, rag_evaluation]",0,0.0,"Этот подход признает критическую важность как качества самой извлеченной информации, так и эффек..."
2,Как происходит индексация?,[rag_paradigms],"[rag_paradigms, rag_paradigms, rag_paradigms]",1,1.0,Включает три основных функциональных этапа: Индексация (Indexing): На этом этапе происходит подг...
3,Какие существуют техники оптимизации запроса?,[rag_paradigms],"[rag_paradigms, rag_paradigms, rag_paradigms]",1,1.0,"Гибридные стратегии поиска: Комбинирование различных методов поиска (например, семантического и ..."
4,С какими основными проблемами и ограничениями сталкивается RAG при практическом применении?,[rag_challenges],"[rag_paradigms, rag_paradigms, rag_challenges]",1,1.0,"Этот подход признает критическую важность как качества самой извлеченной информации, так и эффек..."
5,В каких областях и приложениях наиболее эффективно применяется технология RAG?,[rag_applications],"[rag_paradigms, rag_applications, rag_evaluation]",1,1.0,"Этот подход признает критическую важность как качества самой извлеченной информации, так и эффек..."
6,Какие показатели качества используются для оценки эффективности RAG-систем?,[rag_evaluation],"[rag_evaluation, rag_evaluation, rag_paradigms]",1,1.0,Поэтому разработка и применение методов для систематического измерения качества функционирования...
7,Какие есть бенчмарки для оценки RAG?,[rag_evaluation],"[rag_evaluation, rag_applications, rag_paradigms]",1,1.0,Поэтому разработка и применение методов для систематического измерения качества функционирования...


In [51]:
total_hit = evaluation_df['hit@3'].mean()
total_recall = evaluation_df['recall@3'].mean()

print(f"Общий hit@3: {total_hit:.2%}")
print(f"Общий recall@3: {total_recall:.2%}")

Общий hit@3: 87.50%
Общий recall@3: 87.50%


In [49]:
export_df = evaluation_df.rename(columns={
    'relevant_doc_ids': 'expected_source',
    'predicted_doc_ids': 'retrieved_sources',
    'hit@3': 'hit_at_k'
})[['query', 'expected_source', 'retrieved_sources', 'hit_at_k']]

export_df.to_csv('artifacts/retrieval_eval.csv', index=False)

### Небольшой эксперимент с параметрами retrieval

In [53]:
comparison_results = []

for size in [3, 5]:
    current_chunks_data = []
    for _, row in df_docs.iterrows():
        text_chunks = split_text_into_chunks(row['text'], chunk_size_sentences=size, overlap_sentences=1)
        for i, chunk_content in enumerate(text_chunks):
            current_chunks_data.append({
                "doc_id": row['doc_id'],
                "chunk_text": chunk_content
            })

    temp_df_chunks = pd.DataFrame(current_chunks_data)

    temp_embeddings = model.encode(temp_df_chunks['chunk_text'].tolist(), normalize_embeddings=True, show_progress_bar=False)
    temp_index = faiss.IndexFlatIP(temp_embeddings.shape[1])
    temp_index.add(temp_embeddings)

    def temp_search(query, k=3):
        q_emb = model.encode([query], normalize_embeddings=True, show_progress_bar=False)
        distances, indices = temp_index.search(q_emb, k)
        res = []
        for idx in indices[0]:
            if idx != -1: res.append(temp_df_chunks.iloc[idx]['doc_id'])
        return res

    hits = 0
    for item in qa_benchmark:
        predicted = set(temp_search(item['query'], k=3))
        if set(item['relevant_doc_ids']).intersection(predicted):
            hits += 1

    comparison_results.append({
        "chunk_size_sentences": size,
        "total_chunks": len(temp_df_chunks),
        "hit@3": hits / len(qa_benchmark)
    })

df_comparison = pd.DataFrame(comparison_results)
display(Markdown("### Сравнение качества retrieval при разных размерах чанка"))
display(df_comparison)

### Сравнение качества retrieval при разных размерах чанка

,chunk_size_sentences,total_chunks,hit@3
0,3,71,0.875
1,5,37,0.750


Увеличение размера чанка с 3 предложений до 5 показало результат хуже.

### Обновление базы знаний и переиндексация

In [54]:
query = "Что такое LLM?"
search_results = search_top_k(query, k=3)
display_results(query, search_results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### Результаты поиска по запросу: 'Что такое LLM?'

,doc_id,chunk_id,chunk_text,score
0,rag_advantages,chunk_06,RAG обеспечивает более экономичный способ адаптации LLMs к новым данным и доменам знаний. Интегр...,0.481104
1,rag_abstract,chunk_01,"Генерация с дополненной выборкой (англ. Retrieval-Augmented Generation, RAG) — это подход, при к...",0.480829
2,rag_abstract,chunk_03,"RAG помогает устранить ограничения LLM, такие как устаревание информации, наличие неточностей ил...",0.470763


---

**Документ:** rag_advantages | **Чанк:** chunk_06 | **Score:** 0.481

...

Генерация с дополненной выборкой (англ. Retrieval-Augmented Generation, RAG) — это подход, при котором генерация ответа большой языковой модели (LLM) осуществляется на основе данных, полученных в результате поиска во внешних источниках (Интернет, корпоративные базы данных и справочники, файлы и другие источники), которые таким образом дополняют выборку обучения модели. RAG-система работает в два основных этапа: сначала происходит извлечение релевантных документов или их частей из внешней базы знаний на основе запроса пользователя, а затем полученная информация подставляется вместе со специальными подсказками, указывающими как модель должна использовать эти данные, в контекст языковой модели для генерации итогового ответа.

---

**Документ:** rag_abstract | **Чанк:** chunk_03 | **Score:** 0.471

RAG помогает устранить ограничения LLM, такие как устаревание информации, наличие неточностей или появления галлюцинаций. Основное преимущество RAG в том, что этот архитектурный шаблон расширяет базу знаний LLM до неограниченных размеров (Интернет), даёт быстрый доступ к специализированным базам знаний или к корпоративным базам данных без необходимости переобучения модели. RAG сочетает в себе сильные стороны традиционных систем поиска информации (поисковые системы и системы управления базами данных) с возможностями больших языковых моделей.

---

In [ ]:
query = "Какие есть ограничения у LLM-агентов?"
search_results = search_top_k(query, k=3)
display_results(query, search_results)

In [55]:
new_docs_found = []
if os.path.exists(data_path):
    for filename in os.listdir(data_path):
        if filename.endswith(".txt") and filename.startswith("llm_"):
            doc_name = os.path.splitext(filename)[0]
            if doc_name not in df_docs['doc_id'].values:
                file_path = os.path.join(data_path, filename)
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()
                    new_docs_found.append({"doc_id": doc_name, "text": content})

if new_docs_found:
    print(f"Найдено новых документов: {len(new_docs_found)}")
    df_docs = pd.concat([df_docs, pd.DataFrame(new_docs_found)], ignore_index=True)

    new_chunks_data = []
    for doc in new_docs_found:
        text_chunks = split_text_into_chunks(doc['text'], chunk_size_sentences=3, overlap_sentences=1)
        for i, chunk_content in enumerate(text_chunks):
            new_chunks_data.append({
                "doc_id": doc['doc_id'],
                "chunk_id": f"chunk_{i+1:02d}",
                "chunk_text": chunk_content
            })

    df_chunks = pd.concat([df_chunks, pd.DataFrame(new_chunks_data)], ignore_index=True)

    print("Переиндексация всей базы...")
    chunk_embeddings = get_embeddings(df_chunks['chunk_text'].tolist())
    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings)

    print(f"Обновлено. Всего документов: {len(df_docs)}, всего чанков в индексе: {index.ntotal}")
else:
    print("Новых документов с префиксом 'llm_' не найдено или они уже в базе.")

Найдено новых документов: 2
Переиндексация всей базы...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Обновлено. Всего документов: 8, всего чанков в индексе: 79


In [56]:
query = "Что такое LLM?"
search_results = search_top_k(query, k=3)
display_results(query, search_results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### Результаты поиска по запросу: 'Что такое LLM?'

,doc_id,chunk_id,chunk_text,score
0,llm_abstract,chunk_01,"Большая языковая модель (БЯМ; англ. large language model, LLM) — языковая модель, основанная на ...",0.609352
1,llm_agents,chunk_01,"БЯМ-агенты (LLM Agents, агенты на основе больших языковых моделей) — программный комплекс, в кот...",0.546212
2,rag_advantages,chunk_06,RAG обеспечивает более экономичный способ адаптации LLMs к новым данным и доменам знаний. Интегр...,0.481104


---

**Документ:** llm_abstract | **Чанк:** chunk_01 | **Score:** 0.609

...

БЯМ-агенты (LLM Agents, агенты на основе больших языковых моделей) — программный комплекс, в котором БЯМ выступает в качестве центрального интеллектуального компонента, дополненного функциональными модулями в виде систем памяти, планирования этапов выполнения и инструментального взаимодействия с внешними ИТ-системами через выполнения кода, запуска подпрограмм, прямого обращения к базам данных или через API-запросы. Архитектурно LLM-агенты могут быть реализованы в различных конфигурациях: от простых одиночных агентов, ориентированных на конкретные задачи, до сложных мультиагентных систем с совместным или конкурентным взаимодействием. Несмотря на значительный потенциал в автоматизации когнитивных задач, LLM-агенты сталкиваются с рядом ограничений, включая проблемы контекстной памяти, непоследовательность результатов и сложности долгосрочного планирования, что обуславливает необходимость тщательного проектирования таких систем с учётом специфики прикладных задач.

---

**Документ:** rag_advantages | **Чанк:** chunk_06 | **Score:** 0.481

RAG обеспечивает более экономичный способ адаптации LLMs к новым данным и доменам знаний. Интеграция с разными моделями, агентами и чат-ботами: RAG интегрируется в любую БЯМ и не зависит только от одного производителя моделей. Интегрируется с LLM-приложениями, включая чат-боты и разговорных агентов, предоставляя им доступ к внешним, свежим, частным или специализированным данным.

---

In [57]:
query = "Какие есть ограничения у LLM-агентов?"
search_results = search_top_k(query, k=3)
display_results(query, search_results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### Результаты поиска по запросу: 'Какие есть ограничения у LLM-агентов?'

,doc_id,chunk_id,chunk_text,score
0,llm_agents,chunk_02,"Несмотря на значительный потенциал в автоматизации когнитивных задач, LLM-агенты сталкиваются с ...",0.695577
1,rag_evaluation,chunk_15,"Оценке подлежит ситуация, при которой LLM предоставляются инструкции с предупреждениями о потенц...",0.483625
2,rag_abstract,chunk_03,"RAG помогает устранить ограничения LLM, такие как устаревание информации, наличие неточностей ил...",0.463659


---

**Документ:** llm_agents | **Чанк:** chunk_02 | **Score:** 0.696

...

Оценке подлежит ситуация, при которой LLM предоставляются инструкции с предупреждениями о потенциальных рисках в полученных данных. Даже когда языковые модели содержат необходимые знания и получают такие предупреждения о рисках, они склонны доверять и отдавать приоритет извлеченной информации над своими существующими знаниями. Тестовый набор бенчмарка RGB для включает примеры, на которые модели могут ответить напрямую, но внешние документы содержат фактические ошибки.

---

**Документ:** rag_abstract | **Чанк:** chunk_03 | **Score:** 0.464

RAG помогает устранить ограничения LLM, такие как устаревание информации, наличие неточностей или появления галлюцинаций. Основное преимущество RAG в том, что этот архитектурный шаблон расширяет базу знаний LLM до неограниченных размеров (Интернет), даёт быстрый доступ к специализированным базам знаний или к корпоративным базам данных без необходимости переобучения модели. RAG сочетает в себе сильные стороны традиционных систем поиска информации (поисковые системы и системы управления базами данных) с возможностями больших языковых моделей.

---

In [78]:
llm_questions = [
    "Что такое LLM?",
    "Какие есть ограничения у LLM-агентов?"
]

comparison_data = []

for query in llm_questions:
    after_search = search_top_k(query, k=3)
    after_sources = after_search['doc_id'].tolist()
    old_chunks_indices = df_chunks[df_chunks['doc_id'].str.startswith('rag_')].index
    q_emb = get_embeddings([query])
    distances, indices = index.search(q_emb, index.ntotal)

    before_sources = []
    for idx in indices[0]:
        doc_id = df_chunks.iloc[idx]['doc_id']
        if doc_id.startswith('rag_'):
            before_sources.append(doc_id)
        if len(before_sources) == 3: break

    changed = set(before_sources) != set(after_sources)

    comparison_data.append({
        "query": query,
        "before_retrieved_sources": before_sources,
        "after_retrieved_sources": after_sources,
        "changed": changed
    })

df_retrieval_comparison = pd.DataFrame(comparison_data)
df_retrieval_comparison.to_csv('artifacts/retrieval_before_after_update.csv', index=False)

print("Сравнение для LLM-вопросов завершено.")
display(df_retrieval_comparison)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Сравнение для LLM-вопросов завершено.


,query,before_retrieved_sources,after_retrieved_sources,changed
0,Что такое LLM?,"[rag_advantages, rag_abstract, rag_abstract]","[llm_abstract, llm_agents, rag_advantages]",True
1,Какие есть ограничения у LLM-агентов?,"[rag_evaluation, rag_abstract, rag_advantages]","[llm_agents, rag_evaluation, rag_abstract]",True


### Mini-RAG

In [74]:
def mini_rag(query: str, k: int = 3) -> Dict:
    search_results = search_top_k(query, k=k)

    context_list = search_results['chunk_text'].tolist()
    context_str = "\n\n".join(context_list)

    answer = context_str

    sources = search_results[['doc_id', 'chunk_id']].to_dict(orient='records')

    return {
        "query": query,
        "answer": answer,
        "context_used": context_str,
        "sources": sources
    }

In [77]:
user_query = "Какие преимущества даёт использование RAG?"
rag_output = mini_rag(user_query, k=3)

display(Markdown(f"### Запрос: {rag_output['query']}"))
display(Markdown(f"**Ответ:** {rag_output['answer']}"))
display(Markdown("**Использованные источники:**"))
for s in rag_output['sources']:
    print(f"- {s['doc_id']} ({s['chunk_id']})")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### Запрос: Какие преимущества даёт использование RAG?

**Ответ:** Этот подход признает критическую важность как качества самой извлеченной информации, так и эффективности её представления для достижения высокой производительности RAG-системы. Modular RAG Modular RAG представляет собой наиболее продвинутую парадигму RAG, предлагающую повышенную адаптивность и универсальность за счет включения разнообразных стратегий улучшения компонентов RAG. Этот подход опирается на принципы Naive и Advanced RAG, но вводит специализированные модули и паттерны взаимодействия между ними.

Использование RAG предоставляет ряд существенных преимуществ для улучшения больших языковых моделей. Повышенная точность и надежность: RAG обеспечивает доступ LLM к самым актуальным и надежным фактам, что значительно снижает вероятность генерации неверного или вводящего в заблуждение контента. За счет обоснования ответов на внешних, проверенных источниках информации, RAG гарантирует, что LLM оперирует наиболее точными данными.

RAG помогает устранить ограничения LLM, такие как устаревание информации, наличие неточностей или появления галлюцинаций. Основное преимущество RAG в том, что этот архитектурный шаблон расширяет базу знаний LLM до неограниченных размеров (Интернет), даёт быстрый доступ к специализированным базам знаний или к корпоративным базам данных без необходимости переобучения модели. RAG сочетает в себе сильные стороны традиционных систем поиска информации (поисковые системы и системы управления базами данных) с возможностями больших языковых моделей.

**Использованные источники:**

- rag_paradigms (chunk_18)
- rag_advantages (chunk_01)
- rag_abstract (chunk_03)


In [76]:
specific_questions = [
    "Какие преимущества даёт использование RAG?",
    "Что такое LLM?",
    "Какие есть ограничения у LLM-агентов?"
]

rag_examples_data = []

for query in specific_questions:
    res = mini_rag(query, k=3)
    rag_examples_data.append({
        "question": res['query'],
        "answer": res['answer'],
        "retrieved_sources": [f"{s['doc_id']} ({s['chunk_id']})" for s in res['sources']]
    })

df_examples = pd.DataFrame(rag_examples_data)
df_examples.to_csv('artifacts/rag_examples.csv', index=False)
display(df_examples)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,question,answer,retrieved_sources
0,Какие преимущества даёт использование RAG?,"Этот подход признает критическую важность как качества самой извлеченной информации, так и эффек...","[rag_paradigms (chunk_18), rag_advantages (chunk_01), rag_abstract (chunk_03)]"
1,Что такое LLM?,"Большая языковая модель (БЯМ; англ. large language model, LLM) — языковая модель, основанная на ...","[llm_abstract (chunk_01), llm_agents (chunk_01), rag_advantages (chunk_06)]"
2,Какие есть ограничения у LLM-агентов?,"Несмотря на значительный потенциал в автоматизации когнитивных задач, LLM-агенты сталкиваются с ...","[llm_agents (chunk_02), rag_evaluation (chunk_15), rag_abstract (chunk_03)]"


### Краткий анализ ошибок

In [79]:
user_query = "Что такое RAG?"
rag_output = mini_rag(user_query, k=3)

display(Markdown(f"### Запрос: {rag_output['query']}"))
display(Markdown(f"**Ответ:** {rag_output['answer']}"))
display(Markdown("**Использованные источники:**"))
for s in rag_output['sources']:
    print(f"- {s['doc_id']} ({s['chunk_id']})")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### Запрос: Что такое RAG?

**Ответ:** Этот подход признает критическую важность как качества самой извлеченной информации, так и эффективности её представления для достижения высокой производительности RAG-системы. Modular RAG Modular RAG представляет собой наиболее продвинутую парадигму RAG, предлагающую повышенную адаптивность и универсальность за счет включения разнообразных стратегий улучшения компонентов RAG. Этот подход опирается на принципы Naive и Advanced RAG, но вводит специализированные модули и паттерны взаимодействия между ними.

Благодаря своей способности сочетать обширные знания больших языковых моделей с актуальной информацией из внешних источников, RAG находит широкое применение в самых разных областях. RAG находит применение во множестве контекстов, включая вопросно-ответные системы, анализ временных рядов с использованием агентных RAG-фреймворков, повышение фактичности в медицинских системах обработки изображений и языка, улучшение точности в юридических и политических приложениях, повышение качества автоматического распознавания речи (ASR), обеспечение многоязычного доступа к информации, а также в качестве инструментов помощи в написании научных работ, таких как LLM-Ref или сервисы типа OpenAI Deep Research. Вопросно-ответные и консультационные системы Одним из наиболее распространенных применений RAG является открытое вопросно-ответное взаимодействие, где системы генерируют ответы на широкий спектр вопросов, извлекая релевантную информацию и обосновывая свои ответы полученными данными.

RAG сочетает в себе сильные стороны традиционных систем поиска информации (поисковые системы и системы управления базами данных) с возможностями больших языковых моделей. С появлением больших языковых моделей исследования RAG первоначально фокусировались на использовании их способности к обучению в контексте, преимущественно затрагивая этап вывода. Последующие работы углубили анализ, постепенно интегрируя методы RAG в процесс тонкой настройки LLM.

**Использованные источники:**

- rag_paradigms (chunk_18)
- rag_applications (chunk_01)
- rag_abstract (chunk_04)


Определение RAG не даёт.

In [80]:
user_query = "Бенчмарки для оценки RAG"
rag_output = mini_rag(user_query, k=3)

display(Markdown(f"### Запрос: {rag_output['query']}"))
display(Markdown(f"**Ответ:** {rag_output['answer']}"))
display(Markdown("**Использованные источники:**"))
for s in rag_output['sources']:
    print(f"- {s['doc_id']} ({s['chunk_id']})")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### Запрос: Бенчмарки для оценки RAG

**Ответ:** Поэтому разработка и применение методов для систематического измерения качества функционирования RAG-систем приобретает особую важность. Для систематической оценки RAG разработан ряд бенчмарк-тестов и инструментов. Эти средства предоставляют количественные метрики, которые не только измеряют производительность моделей, но и способствуют детальному анализу их возможностей в различных аспектах оценки.

Разработанная система Tc-Rag, представляющая собой Turing-Complete RAG, показывает превосходные результаты в медицинских вопросно-ответных задачах, что подчеркивает важность RAG для обеспечения точности и актуальности информации в этих областях. Персонализированные рекомендации и реферирование документов RAG также успешно применяется в системах персонализированных рекомендаций, где извлекаются предпочтения пользователей или история их взаимодействия для генерации индивидуальных предложений. В области автоматического реферирования документов RAG используется для создания кратких изложений, основанных на извлеченных знаниях, что позволяет быстро получать ключевую информацию из больших объёмов текста.

Этот подход признает критическую важность как качества самой извлеченной информации, так и эффективности её представления для достижения высокой производительности RAG-системы. Modular RAG Modular RAG представляет собой наиболее продвинутую парадигму RAG, предлагающую повышенную адаптивность и универсальность за счет включения разнообразных стратегий улучшения компонентов RAG. Этот подход опирается на принципы Naive и Advanced RAG, но вводит специализированные модули и паттерны взаимодействия между ними.

**Использованные источники:**

- rag_evaluation (chunk_02)
- rag_applications (chunk_05)
- rag_paradigms (chunk_18)


Названия самих бенчмарков не приводит.

In [81]:
user_query = "Приведи примеры LLM-агентов?"
rag_output = mini_rag(user_query, k=3)

display(Markdown(f"### Запрос: {rag_output['query']}"))
display(Markdown(f"**Ответ:** {rag_output['answer']}"))
display(Markdown("**Использованные источники:**"))
for s in rag_output['sources']:
    print(f"- {s['doc_id']} ({s['chunk_id']})")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### Запрос: Приведи примеры LLM-агентов?

**Ответ:** Несмотря на значительный потенциал в автоматизации когнитивных задач, LLM-агенты сталкиваются с рядом ограничений, включая проблемы контекстной памяти, непоследовательность результатов и сложности долгосрочного планирования, что обуславливает необходимость тщательного проектирования таких систем с учётом специфики прикладных задач. Примеры LLM-агентов из исследований: диалоговые системы психологической поддержки, симуляторы экономического поведения, виртуальные города с агентами (Generative Agents, AgentSims), системы для прогнозирования судебных решений, ассистенты для научных исследований, агент для химии, предназначенный для выполнения задач в областях органического синтеза, разработки лекарств и проектирования материалов (ChemCrow), математические помощники (Math Agents), образовательные системы (EduChat, CodeHelp) и другие. Microsoft Copilot, работая на основе моделей GPT-4, DALL-E 3 и Prometheus, функционирует как ИИ-агент, интегрированный с продуктами Microsoft 365, Windows и GitHub, автоматизирующий рабочие процессы и генерирующий контент с возможностью создания пользовательских решений через Copilot Studio.

БЯМ-агенты (LLM Agents, агенты на основе больших языковых моделей) — программный комплекс, в котором БЯМ выступает в качестве центрального интеллектуального компонента, дополненного функциональными модулями в виде систем памяти, планирования этапов выполнения и инструментального взаимодействия с внешними ИТ-системами через выполнения кода, запуска подпрограмм, прямого обращения к базам данных или через API-запросы. Архитектурно LLM-агенты могут быть реализованы в различных конфигурациях: от простых одиночных агентов, ориентированных на конкретные задачи, до сложных мультиагентных систем с совместным или конкурентным взаимодействием. Несмотря на значительный потенциал в автоматизации когнитивных задач, LLM-агенты сталкиваются с рядом ограничений, включая проблемы контекстной памяти, непоследовательность результатов и сложности долгосрочного планирования, что обуславливает необходимость тщательного проектирования таких систем с учётом специфики прикладных задач.

Оценке подлежит ситуация, при которой LLM предоставляются инструкции с предупреждениями о потенциальных рисках в полученных данных. Даже когда языковые модели содержат необходимые знания и получают такие предупреждения о рисках, они склонны доверять и отдавать приоритет извлеченной информации над своими существующими знаниями. Тестовый набор бенчмарка RGB для включает примеры, на которые модели могут ответить напрямую, но внешние документы содержат фактические ошибки.

**Использованные источники:**

- llm_agents (chunk_02)
- llm_agents (chunk_01)
- rag_evaluation (chunk_15)


Хороший ответ в первом же текстовом фрагменте.